# Iteration 4 — NHS Adult Autism Assessment DES

**Discrete-event simulation (DES)** of the NHS adult autism assessment pathway, built with **SimPy**.

> **How to use this notebook**  
> Run every cell **top to bottom** (`Kernel → Restart & Run All`).  
> Edit scenario parameters in **Section 2**, then re-run from **Section 12** onward.

---

## Table of contents

| Part | Sections | What you will find |
|------|----------|-------------------|
| **A — Background** | Intro | Problem, approach, assumptions, limitations |
| **B — Build the model** | 1–11 | Parameters, classes, pathway logic, capacity engine |
| **C — Run the simulation** | 12–13 | `single_run()` and `multiple_runs()` |
| **D — Results & scenarios** | 14–16 | Baseline KPIs and warm-up sensitivity |
| **E — Test the model** | 17–21 | Boundary tests, RNG checks, automated V&V |
| **F — Next steps** | 22 | Calibration and experiment ideas |

---

## Part A — Background

### What problem are we solving?

NHS autism assessment services face **long referral-to-diagnosis waits**, often above the **18-week (126-day) RTT target**. The service must balance:

- **Demand** — weekday referrals arriving over years
- **Capacity** — limited clinician hours at each stage
- **Pathway complexity** — triage, further assessment, post-diagnosis support, review loops

**Questions this model helps answer:**

1. How long does a referred patient wait for diagnosis at current capacity?
2. Which stage is the bottleneck (longest queue)?
3. How sensitive are waits to **starting backlog** (warm-up)?
4. What happens if capacity drops or demand doubles?

This is a **stochastic** model — randomness in arrivals, durations, and branching means each replication differs. We use **fixed seeds** and **multiple replications** for reproducibility and uncertainty.

### How are we solving it?

| Step | Method |
|------|--------|
| 1 | Each patient is a SimPy **process** through triage → 7 clinical stages → diagnosis → support → review |
| 2 | `WorkforceHoursResource` limits **clinician hours per weekday** (not fixed slots) |
| 3 | Random inputs: exponential arrivals, triangular durations, Bernoulli branching — 25 independent RNG streams |
| 4 | KPI **cohort** = referrals in the collection window only; warm-up builds backlog but is excluded from RTT |
| 5 | Three phases: **warm-up → collection → drain** |
| 6 | Automated **V&V tests** verify logic, conservation, and capacity stress response |

| Component | Role |
|-----------|------|
| `Experiment` | Parameters, RNG, flow counters |
| `Audit` | RTT, wait, queue, utilisation KPIs |
| `Patient` | SimPy process through the pathway |
| `WorkforceHoursResource` | Weekday hour capacity + priority queues |
| `AutismPathwaySystem` | Resources + weekday referral generator |
| `single_run()` | One replication through all three phases |


### Assumptions & limitations

Change parameters in **Section 2** to test alternatives.

#### Assumptions

| Area | Assumption |
|------|------------|
| **Demand** | Constant weekday rate; exponential inter-arrivals; no weekend referrals |
| **Capacity** | Fixed hours per weekday per stage; Mon–Fri only; no leave or holidays |
| **Durations** | Triangular `[min, mode, max]` hours — placeholders, not yet calibrated |
| **Branching** | Memoryless Bernoulli at each gate (e.g. 20% non-diagnosis at assessment) |
| **Priority** | Re-drawn at each stage (~10–15%) — not a persistent urgent flag |
| **Triage & outcome** | Instant (no queue) |
| **Calendar** | `env.now % 7` for weekday; not a real calendar date |
| **Review loop** | Capped at `MAX_REVIEW_LOOPS` (50) |

#### Limitations

| Limitation | Impact |
|------------|--------|
| **Not calibrated** | Absolute RTT values are illustrative until fitted to NHS/local data |
| **No case persistence** | Complexity at one stage does not affect later stages |
| **Single pooled team** | No named staff or skill mix |
| **Homogeneous service** | No regional or provider variation |
| **Code in notebook** | Harder to unit-test than a Python package (see **Section 22**) |
| **Zero diagnoses** | If no one completes diagnosis, RTT reports `0.0` (empty sample) |

> **Warm-up note**  
> Default `WARMUP_DAYS = 730` (2 years) simulates a service already under pressure.  
> Set `WARMUP_DAYS = 0` for an empty-start service. Compare both in **Section 16**.


---

## Part B — Build the model

Sections **1–11** define the full simulation: parameters, KPI collection, experiment config, patient logic, capacity engine, and system orchestrator.

| Section | Topic |
|---------|--------|
| 1 | Imports |
| 2 | Global parameters *(edit scenarios here)* |
| 3–4 | Trace utility and distributions |
| 5–7 | Capacity helpers, Audit, Experiment |
| **8** | **Patient pathway** *(read before Section 9)* |
| 9 | Patient class + flow diagrams (9.1) |
| 10–11 | WorkforceHoursResource and AutismPathwaySystem |

> **Recommended reading order:** Section 8 → Section 9 → Sections 10–11.


## 1. Imports

**Purpose:** Load libraries and distribution classes used by the model.

| Library | Role |
|---------|------|
| **SimPy** | Discrete-event simulation engine |
| **NumPy / pandas** | KPI aggregation and tables |
| **joblib** | Parallel replications |

Distributions are imported from `adhd_simpy.Model.distributions` in **Section 4**.


In [44]:
from __future__ import annotations

import copy
import itertools
import secrets
import warnings
from collections import deque

import numpy as np
import pandas as pd
import simpy
from joblib import Parallel, delayed


## 2. Global parameters

**Purpose:** Single place to edit the scenario. Re-run from **Section 12** after changes.

| Group | Key variables | Controls |
|-------|---------------|----------|
| **Timing** | `WARMUP_DAYS`, `RUN_LENGTH`, `MAX_DRAIN_DAYS` | Warm-up, collection window, drain cap |
| **Demand** | `REFERRALS_PER_DAY` | Weekday referral rate |
| **Durations** | `DURATION_*` | Triangular service times (hours) |
| **Branching** | `PCT_*` | Exit and route probabilities |
| **Capacity** | `WORKFORCE_HOURS_*` | Clinician hours per weekday |
| **Priority** | `PCT_PRIORITY_*` | Share entering priority queue |
| **RNG** | `N_STREAMS`, `DEFAULT_RND_SET`, `N_REP` | Reproducible random streams |


In [ ]:
from adhd_simpy.Model import (
    Audit,
    AutismPathwaySystem,
    Experiment,
    Patient,
    WorkforceHoursResource,
    multiple_runs,
    single_run,
)
from adhd_simpy.Model.parameters import *
from adhd_simpy.Model.utils import derive_daily_slots, triangular_mean_hours, trace
from adhd_simpy.Model.verification import (
    assert_monotonic_increasing,
    run_demand_stress_verification,
    run_flow_conservation_verification,
    run_math_convergence_verification,
    run_rtt_cohort_verification,
    run_seed_verification,
    triangular_mean_days,
)


## 3. Trace utility

**Purpose:** Optional debug printing for short test runs.

Set `TRACE = True` in **Section 2**, then run **Section 14** to follow individual patients through the pathway. Set `TRACE = False` when finished.


## 4. Distributions

**Purpose:** Stochastic inputs for arrivals, durations, and branching.

Each distribution is seeded independently via `Experiment.init_sampling()` so replications are reproducible.

| Distribution | Used for |
|--------------|----------|
| `Exponential` | Inter-arrival time between weekday referrals |
| `Triangular` | Appointment durations (hours → days) |
| `Bernoulli` | Branching gates (reject, discharge, diagnosis, priority) |
| `Choice` | Review outcome (continue / formal discharge / self-removal) |


## 5. Capacity helpers

**Purpose:** Reporting helpers only — **not** used for scheduling.

The simulation binds capacity through `WorkforceHoursResource`, not slot counts.

`derive_daily_slots()` divides workforce hours by mean appointment duration to estimate an upper bound on daily starts (sanity checks only).


## 6. Audit class

**Purpose:** Collect **Key Performance Indicators (KPIs)** during simulation.

#### Cohort filtering

Primary RTT and wait metrics include only patients referred during the **collection window** `[collection_start, collection_end)`.

Warm-up referrals are tracked separately in `*_LEGACY` and `FLOW_WARMUP_REFERRAL_*` keys.

#### Key outputs from `summarize()`

| Prefix | Examples |
|--------|----------|
| `ACCESS_*` | Referral-to-stage RTT and stage wait times |
| `QUEUE_*` | Queue length snapshots |
| `CAPACITY_*` | Utilisation by stage |
| `COHORT_RTT_VALID` | `True` when all cohort patients drained with zero backlog |


## 7. Experiment class

**Purpose:** Central configuration — parameters, RNG, and flow counters.

| Holds | Examples |
|-------|----------|
| Branching probabilities | `PCT_*` → Bernoulli distributions |
| Duration triplets | `DURATION_*` → Triangular distributions |
| Workforce hours | `WORKFORCE_HOURS_*` → `derived_capacity` metadata |
| RNG streams | 25 independent streams via `init_sampling()` |
| Flow counters | `results` dict updated by `Patient` processes |

**Scenario override example:**
```python
Experiment(auditor=Audit(), workforce_hours_assessment=20)
```


## 8. Patient pathway — how patients flow

**Purpose:** Conceptual map of `Patient.process()` before reading the code.

Each patient is a **SimPy process**. `collect_stats = True` only for referrals in the **collection window**; warm-up patients traverse the pathway but are excluded from cohort KPIs.

> Branching probabilities are set in **Section 2** (`PCT_*`) — not shown on the diagram.

#### Pathway (text)

```
Referral (weekday only)
    │
    ▼
 Triage ──reject──► EXIT
    │ accept
    ▼
 Screening ──discharge──► EXIT
    │ pass
    ▼
 Pre-assessment ──reject──► EXIT
    │ pass
    ▼
 Assessment ──non-diagnosis──► EXIT
    │ pass
    ├──yes──► Further assessment ──non-diagnosis──► EXIT
    │              │ pass
    └──no───┬──────┘
            ▼
 Diagnostic outcome ──non-diagnosis──► EXIT
    │ diagnosis confirmed → record RTT("diagnosis")
    ▼
 Post-diag support (clinical / other)
    ▼
 Review loop (up to MAX_REVIEW_LOOPS)
    ├── formal discharge ──► EXIT
    ├── self-removal ──► EXIT
    └── continue ──► back to post-diag support
```

#### Stage parameters

| Stage | Resource | Branch parameter |
|-------|----------|------------------|
| Screening | `screening_resource` | `PCT_SCREENING_DISCHARGED` |
| Pre-assessment | `pre_assessment_resource` | `PCT_PRE_ASS_REJECTED` |
| Assessment | `assessment_resource` | `PCT_NON_DIAGNOSIS_AT_ASSESSMENT` |
| Further assessment | `further_assessment_resource` | `PCT_NEEDS_FURTHER_ASSESSMENT` / `PCT_NON_DIAGNOSIS_AT_FURTHER_ASSESSMENT` |
| Post-diag clinical / other | respective resources | `PCT_POST_DIAG_CLINICAL` |
| Review | `review_resource` | `PCT_REVIEW_*` |

**Non-capacity stages:** triage and diagnostic outcome (instant Bernoulli).

**Clinical stage subprocess:** sample duration → join priority/standard queue → wait for hours → serve → `notify_service_complete()`.

#### Iteration 4 vs earlier versions

| Aspect | Earlier iterations | Iteration 4 |
|--------|-------------------|-------------|
| Capacity | Fixed slots/day | **Workforce hours/day** |
| Scheduling | One slot = one patient | **Best-fit** in remaining hours |
| Priority | Optional | Priority deque served first |
| KPI cohort | All referrals | **`collect_stats`** filter |

> **Visual diagrams:** see **Section 9.1** for publication-quality flowcharts.


## 9. Patient class

**Purpose:** SimPy implementation of the pathway described in **Section 8**.

| Method | Role |
|--------|------|
| `process()` | Main generator — full pathway from referral to discharge |
| `_clinical_stage()` | Queue → serve → complete one capacity-constrained stage |
| `_wait_for_slot()` | Request workforce hours; record wait KPI |
| `_diagnostic_outcome()` | Non-capacity diagnosis decision |
| `_review_and_support_loop()` | Post-diagnosis review cycle |

Flow counters use `_inc(key)` — cohort counters increment only when `collect_stats` is `True`.

> Run the code cell below to load the class, then see **Section 9.1** for flow diagrams.


### 9.1 Patient flow diagrams

**Purpose:** Visual summary of `Patient.process()` and subprocesses.

| Figure | File | Shows |
|--------|------|-------|
| **Fig. 1** | `01_patient_pathway.png` / `.svg` | Full pathway in three phases |
| **Fig. 2** | `02_clinical_stage.png` / `.svg` | Capacity-constrained stage subprocess |
| **Fig. 3** | `03_review_loop.png` / `.svg` | Review and support loop |

- **300 DPI PNG** — for Word/PowerPoint
- **SVG** — vector format; stays sharp when zoomed
- **No fixed probabilities** on figures — edit `PCT_*` in Section 2

#### Figure 1 — Main pathway
![Figure 1: Patient pathway](assets/iteration4/01_patient_pathway.png)

#### Figure 2 — Clinical stage subprocess
![Figure 2: Clinical stage](assets/iteration4/02_clinical_stage.png)

#### Figure 3 — Review loop
![Figure 3: Review loop](assets/iteration4/03_review_loop.png)

> Run the cell below to regenerate figures after model changes.


In [ ]:
from pathlib import Path
import sys
from IPython.display import SVG, display, Markdown

_assets = Path('assets/iteration4').resolve()
if str(_assets) not in sys.path:
    sys.path.insert(0, str(_assets))
from generate_pathway_diagrams import generate_pathway_diagrams

FLOW_ASSET_DIR = Path('assets/iteration4')
diagram_paths = generate_pathway_diagrams(FLOW_ASSET_DIR, also_svg=True)

figs = [
    ('Figure 1 — Main pathway', '01_patient_pathway'),
    ('Figure 2 — Clinical stage subprocess', '02_clinical_stage'),
    ('Figure 3 — Review loop', '03_review_loop'),
]
for title, stem in figs:
    display(Markdown(f'#### {title} *(SVG — zoom without blur)*'))
    display(SVG(filename=str(FLOW_ASSET_DIR / f'{stem}.svg')))

print('Saved PNG (300 DPI) + SVG to:', FLOW_ASSET_DIR.resolve())


## 10. WorkforceHoursResource — capacity engine

**Purpose:** Enforce weekday clinician-hour capacity with priority queues.

| Feature | Behaviour |
|---------|-----------|
| Capacity | Clinician **hours per weekday** (Mon–Fri; weekends = 0) |
| Queues | Priority deque served before standard |
| Scheduling | **Best-fit** — largest job that fits remaining hours |
| Accounting | Tracks released / used / unused hours |
| Validation | `final_validate()` checks balance at run end |


## 11. AutismPathwaySystem

**Purpose:** Orchestrator — wires seven stage resources and the weekday referral generator.

| Responsibility | Detail |
|----------------|--------|
| Resources | One `WorkforceHoursResource` per clinical stage |
| Arrivals | Exponential inter-arrivals on weekdays until `arrival_stop` |
| Collection window | Passed to Audit and Patient for cohort filtering |


---

## Part C — Run the simulation

Each `single_run()` executes **three phases**:

```
Phase 1  WARM-UP     Days 0 → WARMUP_DAYS           KPI counters OFF
Phase 2  COLLECTION  Days WARMUP_DAYS → + RUN_LENGTH KPI cohort ON
Phase 3  DRAIN       After last referral until empty (max MAX_DRAIN_DAYS)
```

When collection ends, referrals stop and the simulation **drains** until every patient has exited (or `MAX_DRAIN_DAYS` is reached).

> **Valid RTT requires** `IN_SYSTEM_END = 0` at run end (`COHORT_RTT_VALID = True`).


## 12. single_run() — one replication

**Purpose:** Run one full replication through warm-up, collection, and drain.

| Returns | Description |
|---------|-------------|
| Flow counters | Arrivals, exits, diagnoses, etc. |
| KPIs | From `Audit.summarize()` |
| Metadata | `WARMUP_DAYS`, `DRAIN_DAYS`, `COHORT_RTT_VALID`, … |

See **Part C** for the three-phase lifecycle.


## 13. multiple_runs() — parallel replications

**Purpose:** Run `N_REP` independent replications (default 20) and return a DataFrame.

| Setting | Effect |
|---------|--------|
| `use_fixed_seed=True` | Rep `i` uses seed `DEFAULT_RND_SET + i` |
| `n_jobs=-1` | Use all CPU cores |
| `warmup_days`, `run_length` | Passed through to `single_run()` |

One row per replication; columns include all KPI fields.


## 14. Quick trace test

**Purpose:** Sanity-check pathway logic on a 3-day run.

1. Set `TRACE = True` in **Section 2**
2. Run this cell — watch patients move through stages
3. Set `TRACE = False` when done


In [56]:
TRACE = True

test_days = 3
test_experiment = Experiment(auditor=Audit())
test_results = single_run(test_experiment, rep=0, run_length=test_days, warmup_days=0)

TRACE = False
print(f"Trace test complete — arrivals: {test_results['ARRIVED_TOTAL']}")


[Time 0.221 | Monday] Patient 1 entered system. Referral submitted.
[Time 0.221 | Monday] Patient 1 Exit - Referral Rejected at Triage.
[Time 0.360 | Monday] Patient 2 entered system. Referral submitted.
[Time 0.360 | Monday] Patient 2 Exit - Referral Rejected at Triage.
[Time 0.499 | Monday] Patient 3 entered system. Referral submitted.
[Time 0.499 | Monday] Patient 3 queued for SCREENING (Queue pos: 0).
[Time 0.499 | Monday] >> Patient 3 officially ENTERED SCREENING stage.
[Time 0.527 | Monday] Patient 3 queued for PRE-ASSESSMENT (Queue pos: 0).
[Time 0.527 | Monday] >> Patient 3 officially ENTERED PRE-ASSESSMENT stage.
[Time 0.578 | Monday] Patient 3 queued for CORE ASSESSMENT (Queue pos: 0).
[Time 0.578 | Monday] >> Patient 3 officially ENTERED CORE ASSESSMENT stage.
[Time 0.740 | Monday] Patient 3 queued for POST-DIAG OTHER SUPPORT.
[Time 0.740 | Monday] >> Patient 3 officially ENTERED POST-DIAG OTHER SUPPORT stage.
[Time 0.809 | Monday] Patient 3 queued for FINAL CASE DISCHARGE R

---

## Part D — Results & scenarios

Baseline KPIs and warm-up sensitivity (**Sections 15–16**).

| Section | What it runs |
|---------|--------------|
| **15** | Baseline — no warm-up vs default warm-up |
| **16.1** | Single-rep warm-up comparison (fast) |
| **16.2** | Multi-rep warm-up comparison with **paired seeds** |

#### Key outputs to check

| Metric | Meaning |
|--------|---------|
| `ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS` | Referral → confirmed diagnosis |
| `QUEUE_PEAK_*` | Maximum waiting-list depth by stage |
| `OVERALL_SYSTEM_UTILISATION` | Clinician hours used vs released |
| `COHORT_RTT_VALID` | All cohort patients drained; zero backlog |


## 15. Main results run

**Purpose:** Print baseline KPIs for the default scenario.

Two cells below compare:

1. **No warm-up** (`warmup_days=0`) — empty service at start
2. **Default warm-up** (`warmup_days=730`) — 2 years of pre-collection demand

Check `COHORT_RTT_VALID` and `ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS` in each output.


In [57]:
TRACE = False

experiment = Experiment(auditor=Audit())
results = single_run(
    experiment,
    rep=0,
    run_length=RUN_LENGTH,
    warmup_days=0
)

print("\nResults:")
print(
    f"Referrals per weekday: {REFERRALS_PER_DAY} | "
    f"Warm-up: {results['WARMUP_DAYS']:.0f}d ({results['WARMUP_DAYS'] / 365:.2f}y) | "
    f"Collection: {results['COLLECTION_WINDOW_DAYS']:.0f}d ({results['COLLECTION_WINDOW_DAYS'] / 365:.2f}y) | "
    f"Drain: {results['DRAIN_DAYS']:.0f}d (max {MAX_DRAIN_DAYS}d) | Cohort drain complete: {results['COHORT_DRAIN_COMPLETE']}"
)
print(f"Simulation ended at day {results['SIM_END_DAYS']:.0f}")
print(f"Warm-up referrals (excluded from KPIs): {results.get('FLOW_WARMUP_REFERRAL_RTT_DIAGNOSIS', 0):.0f} diagnosis events during warm-up")

print("\nDerived weekday slots (reporting only):")
for stage, cap in experiment.derived_capacity.items():
    print(f"  {stage}: {cap['derived_slots_per_day']} slots/day ({cap['workforce_hours_per_day']:.1f}h)")

print(f"\nTotal Arrivals: {results['ARRIVED_TOTAL']}")
print(f"Total Exits: {results['EXIT_TOTAL']}")
print(f"Total In System at End: {results['IN_SYSTEM_END']} (must be 0 for valid cohort RTT)")
print(f"Cohort RTT valid (zero backlog): {results['COHORT_RTT_VALID']}")
print(f"Total Clinically Completed: {results['CLINICAL_COMPLETED_TOTAL']}")
print(f"Priority queue events: {results['FLOW_PRIORITY_TOTAL']}")

print(f"\nReferral to Pre-Assessment RTT (days): {results['ACCESS_REFERRAL_TO_PRE_ASSESSMENT_RTT_DAYS']:.2f}")
print(f"Referral to Assessment RTT (days): {results['ACCESS_REFERRAL_TO_ASSESSMENT_RTT_DAYS']:.2f}")
print(f"Referral to Diagnosis RTT (days): {results['ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS']:.2f}")
print(f"Pre-Assessment Wait (days): {results['ACCESS_PRE_ASSESSMENT_WAIT_DAYS']:.2f}")
print(f"Total System Backlog at End: {results['QUEUE_TOTAL_BACKLOG']:.0f}")
print(f"Overall Capacity Utilisation: {results['OVERALL_SYSTEM_UTILISATION']:.2f}%")
print(f"Diagnosis Rate among Arrivals: {results['FLOW_DIAGNOSIS_RATE_PCT']:.2f}%")




Results:
Referrals per weekday: 5.0 | Warm-up: 0d (0.00y) | Collection: 1825d (5.00y) | Cool-down: 365d | Drain extension: 0d
Simulation ended at day 2190
Warm-up referrals (excluded from KPIs): 0 diagnosis events during warm-up

Derived weekday slots (reporting only):
  screening: 6 slots/day (5.0h)
  pre_assessment: 9 slots/day (10.0h)
  assessment: 5 slots/day (24.0h)
  further_assessment: 3 slots/day (10.0h)
  post_diag_clinical: 5 slots/day (8.0h)
  post_diag_other: 3 slots/day (5.0h)
  review: 10 slots/day (5.0h)

Total Arrivals: 6609
Total Exits: 6609
Total In System at End: 0 (must be 0 for valid cohort RTT)
Cohort RTT valid (zero backlog): True
Total Clinically Completed: 2966
Priority queue events: 2115

Referral to Pre-Assessment RTT (days): 0.13
Referral to Assessment RTT (days): 0.35
Referral to Diagnosis RTT (days): 0.42
Pre-Assessment Wait (days): 0.01
Total System Backlog at End: 0
Overall Capacity Utilisation: 34.55%
Diagnosis Rate among Arrivals: 44.88%


In [58]:
TRACE = False

experiment = Experiment(auditor=Audit())
results = single_run(
    experiment,
    rep=0,
    run_length=RUN_LENGTH,
    warmup_days=WARMUP_DAYS
)

print("\nResults:")
print(
    f"Referrals per weekday: {REFERRALS_PER_DAY} | "
    f"Warm-up: {results['WARMUP_DAYS']:.0f}d ({results['WARMUP_DAYS'] / 365:.2f}y) | "
    f"Collection: {results['COLLECTION_WINDOW_DAYS']:.0f}d ({results['COLLECTION_WINDOW_DAYS'] / 365:.2f}y) | "
    f"Drain: {results['DRAIN_DAYS']:.0f}d (max {MAX_DRAIN_DAYS}d) | Cohort drain complete: {results['COHORT_DRAIN_COMPLETE']}"
)
print(f"Simulation ended at day {results['SIM_END_DAYS']:.0f}")
print(f"Warm-up referrals (excluded from KPIs): {results.get('FLOW_WARMUP_REFERRAL_RTT_DIAGNOSIS', 0):.0f} diagnosis events during warm-up")

print("\nDerived weekday slots (reporting only):")
for stage, cap in experiment.derived_capacity.items():
    print(f"  {stage}: {cap['derived_slots_per_day']} slots/day ({cap['workforce_hours_per_day']:.1f}h)")

print(f"\nTotal Arrivals: {results['ARRIVED_TOTAL']}")
print(f"Total Exits: {results['EXIT_TOTAL']}")
print(f"Total In System at End: {results['IN_SYSTEM_END']} (must be 0 for valid cohort RTT)")
print(f"Cohort RTT valid (zero backlog): {results['COHORT_RTT_VALID']}")
print(f"Total Clinically Completed: {results['CLINICAL_COMPLETED_TOTAL']}")
print(f"Priority queue events: {results['FLOW_PRIORITY_TOTAL']}")

print(f"\nReferral to Pre-Assessment RTT (days): {results['ACCESS_REFERRAL_TO_PRE_ASSESSMENT_RTT_DAYS']:.2f}")
print(f"Referral to Assessment RTT (days): {results['ACCESS_REFERRAL_TO_ASSESSMENT_RTT_DAYS']:.2f}")
print(f"Referral to Diagnosis RTT (days): {results['ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS']:.2f}")
print(f"Pre-Assessment Wait (days): {results['ACCESS_PRE_ASSESSMENT_WAIT_DAYS']:.2f}")
print(f"Total System Backlog at End: {results['QUEUE_TOTAL_BACKLOG']:.0f}")
print(f"Overall Capacity Utilisation: {results['OVERALL_SYSTEM_UTILISATION']:.2f}%")
print(f"Diagnosis Rate among Arrivals: {results['FLOW_DIAGNOSIS_RATE_PCT']:.2f}%")




Results:
Referrals per weekday: 5.0 | Warm-up: 730d (2.00y) | Collection: 1825d (5.00y) | Cool-down: 365d | Drain extension: 0d
Simulation ended at day 2920
Warm-up referrals (excluded from KPIs): 1 diagnosis events during warm-up

Derived weekday slots (reporting only):
  screening: 6 slots/day (5.0h)
  pre_assessment: 9 slots/day (10.0h)
  assessment: 5 slots/day (24.0h)
  further_assessment: 3 slots/day (10.0h)
  post_diag_clinical: 5 slots/day (8.0h)
  post_diag_other: 3 slots/day (5.0h)
  review: 10 slots/day (5.0h)

Total Arrivals: 6630
Total Exits: 6630
Total In System at End: 0 (must be 0 for valid cohort RTT)
Cohort RTT valid (zero backlog): True
Total Clinically Completed: 2990
Priority queue events: 2155

Referral to Pre-Assessment RTT (days): 0.13
Referral to Assessment RTT (days): 0.35
Referral to Diagnosis RTT (days): 0.42
Pre-Assessment Wait (days): 0.01
Total System Backlog at End: 0
Overall Capacity Utilisation: 35.03%
Diagnosis Rate among Arrivals: 45.10%


## 16. Warm-up comparison

**Purpose:** Isolate the effect of starting backlog on waits and queues.

Compare **with warm-up** vs **without warm-up** using the same scenario parameters and RNG seeds.

| Subsection | Method | Best for |
|------------|--------|----------|
| **16.1** | Single replication (`rep=0`) | Quick snapshot |
| **16.2** | `multiple_runs()` — `N_REP` reps | Mean, spread, paired RTT delta |

Both use `use_fixed_seed=True` so replication `i` draws the **same stochastic stream** in each scenario — only warm-up length differs.


### 16.1 Single replication comparison

**Purpose:** Fast side-by-side on a **1-year collection window**.

Run the cell below for a quick warm-up vs no-warm-up table.


In [59]:
COMPARE_RUN_LENGTH = 365  # 1-year collection window for quick comparison
COMPARE_WARMUP = 365      # 1-year warm-up (use WARMUP_DAYS for full production runs)

scenarios = [
    ("With warm-up", COMPARE_WARMUP),
    ("Without warm-up", 0),
]

print("=" * 72)
print(" WARM-UP vs NO WARM-UP COMPARISON")
print(f" Collection window: {COMPARE_RUN_LENGTH} days each scenario")
print("=" * 72)

rows = []
for label, warmup in scenarios:
    exp = Experiment(auditor=Audit(), use_fixed_seed=True, random_number_set=42)
    r = single_run(
        exp,
        rep=0,
        warmup_days=warmup,
        run_length=COMPARE_RUN_LENGTH,
        max_drain_days=1825,
    )
    rows.append(
        {
            "Scenario": label,
            "warmup_days": warmup,
            "ARRIVED_ALL": r["ARRIVED_ALL"],
            "ARRIVED_TOTAL": r["ARRIVED_TOTAL"],
            "Diagnosis_RTT_d": r["ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS"],
            "Peak_queue": r["QUEUE_PEAK_ANY_STAGE"],
            "Utilisation_pct": r["OVERALL_SYSTEM_UTILISATION"],
            "COHORT_RTT_VALID": r["COHORT_RTT_VALID"],
            "Warmup_diag_events": r.get("FLOW_WARMUP_REFERRAL_RTT_DIAGNOSIS", 0),
        }
    )

df_compare = pd.DataFrame(rows)
print(df_compare.to_string(index=False))

print("\nInterpretation:")
print("  • With warm-up: ARRIVED_ALL > ARRIVED_TOTAL; queues build before collection.")
print("  • Without warm-up: ARRIVED_ALL == ARRIVED_TOTAL; empty start at day 0.")
print("  • Diagnosis RTT and peak queue typically higher with warm-up (existing backlog).")
print("  • Set WARMUP_DAYS=0 in Section 2 for all runs to use empty-start mode.")



 WARM-UP vs NO WARM-UP COMPARISON
 Collection window: 365 days each scenario
       Scenario  warmup_days  ARRIVED_ALL  ARRIVED_TOTAL  Diagnosis_RTT_d  Peak_queue  Utilisation_pct  COHORT_RTT_VALID  Warmup_diag_events
   With warm-up          365         2609           1302         0.441504         7.0        27.268749              True                   0
Without warm-up            0         1307           1307         0.421237         4.0        27.204496              True                   0

Interpretation:
  • With warm-up: ARRIVED_ALL > ARRIVED_TOTAL; queues build before collection.
  • Without warm-up: ARRIVED_ALL == ARRIVED_TOTAL; empty start at day 0.
  • Diagnosis RTT and peak queue typically higher with warm-up (existing backlog).
  • Set WARMUP_DAYS=0 in Section 2 for all runs to use empty-start mode.


### 16.2 Multi-replication comparison (same seeds)

**Purpose:** Statistical comparison with paired replications.

| Setting | Value |
|---------|-------|
| Replications | `MULTI_REP_N` (default `N_REP`) |
| Seed for rep `i` | `DEFAULT_RND_SET + i` in **both** scenarios |
| What varies | `warmup_days` only (0 vs `WARMUP_DAYS`) |

**Outputs:** summary statistics and a **paired delta** (warm-up − no warm-up) per replication.

Positive delta → warm-up increased diagnosis RTT for that rep.


In [60]:
MULTI_REP_N = N_REP          # set to e.g. 6 for a faster notebook run
MULTI_RUN_LENGTH = RUN_LENGTH
MULTI_WARMUP = WARMUP_DAYS

KPI_COLS = [
    "ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS",
    "ACCESS_REFERRAL_TO_ASSESSMENT_RTT_DAYS",
    "QUEUE_PEAK_ANY_STAGE",
    "OVERALL_SYSTEM_UTILISATION",
    "FLOW_DIAGNOSIS_RATE_PCT",
    "COHORT_RTT_VALID",
]

base_exp = Experiment(
    auditor=Audit(),
    use_fixed_seed=True,
    random_number_set=DEFAULT_RND_SET,
)

print("=" * 72)
print(" MULTI-REPLICATION WARM-UP vs NO WARM-UP (same seeds per rep)")
print(f" Replications: {MULTI_REP_N} | Collection: {MULTI_RUN_LENGTH}d | Warm-up: 0 vs {MULTI_WARMUP}d")
print(f" Base seed: {DEFAULT_RND_SET} (rep i uses seed {DEFAULT_RND_SET} + i)")
print("=" * 72)

df_no_warmup = multiple_runs(
    copy.deepcopy(base_exp),
    n_reps=MULTI_REP_N,
    warmup_days=0,
    run_length=MULTI_RUN_LENGTH,
    n_jobs=-1,
    use_fixed_seed=True,
)
df_with_warmup = multiple_runs(
    copy.deepcopy(base_exp),
    n_reps=MULTI_REP_N,
    warmup_days=MULTI_WARMUP,
    run_length=MULTI_RUN_LENGTH,
    n_jobs=-1,
    use_fixed_seed=True,
)

# --- Summary statistics ---
summary_rows = []
for label, df in [("No warm-up", df_no_warmup), (f"Warm-up ({MULTI_WARMUP}d)", df_with_warmup)]:
    row = {"Scenario": label, "n_reps": len(df)}
    for col in KPI_COLS:
        if col in df.columns:
            if col == "COHORT_RTT_VALID":
                row[col] = df[col].mean()
            else:
                row[f"{col}_mean"] = df[col].mean()
                row[f"{col}_std"] = df[col].std()
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
print("\nSummary (mean ± std across replications):")
display_cols = [c for c in df_summary.columns if c not in ("Scenario", "n_reps")]
print(df_summary[["Scenario", "n_reps"] + display_cols[:6]].to_string(index=False))

# --- Paired comparison (same rep index, same RNG stream) ---
paired = pd.DataFrame({"Replication": df_no_warmup["Replication"].values})
for col in KPI_COLS:
    if col in df_no_warmup.columns:
        paired[f"{col}_no_wu"] = df_no_warmup[col].values
        paired[f"{col}_wu"] = df_with_warmup[col].values
        if col != "COHORT_RTT_VALID":
            paired[f"{col}_delta"] = df_with_warmup[col].values - df_no_warmup[col].values

rtt_col = "ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS"
print(f"\nPaired diagnosis RTT delta (warm-up − no warm-up) per replication:")
print(paired[["Replication", f"{rtt_col}_no_wu", f"{rtt_col}_wu", f"{rtt_col}_delta"]].to_string(index=False, float_format=lambda x: f"{x:.1f}"))

print(f"\nMean paired RTT delta: {paired[f'{rtt_col}_delta'].mean():.1f} days")
print(f"All deltas > 0 (warm-up worse): {(paired[f'{rtt_col}_delta'] > 0).all()}")
print(f"Cohort RTT valid — no warm-up: {df_no_warmup['COHORT_RTT_VALID'].all()}")
print(f"Cohort RTT valid — with warm-up: {df_with_warmup['COHORT_RTT_VALID'].all()}")

print("\nInterpretation:")
print("  • Same rep index → same arrivals/branching; only starting backlog differs.")
print("  • Positive RTT delta → warm-up increased waits for that replication.")
print("  • Compare std across scenarios to see if warm-up adds uncertainty.")


 MULTI-REPLICATION WARM-UP vs NO WARM-UP (same seeds per rep)
 Replications: 20 | Collection: 1825d | Warm-up: 0 vs 730d
 Base seed: 42 (rep i uses seed 42 + i)

Summary (mean ± std across replications):
      Scenario  n_reps  ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS_mean  ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS_std  ACCESS_REFERRAL_TO_ASSESSMENT_RTT_DAYS_mean  ACCESS_REFERRAL_TO_ASSESSMENT_RTT_DAYS_std  QUEUE_PEAK_ANY_STAGE_mean  QUEUE_PEAK_ANY_STAGE_std
    No warm-up      20                                    0.425912                                   0.011867                                     0.351670                                    0.010623                       7.85                  1.631112
Warm-up (730d)      20                                    0.425150                                   0.010298                                     0.350575                                    0.011016                       8.00                  1.521772

Paired diagnosis RTT delta (warm-up − n

---

## Part E — Test the stochastic model

Structured verification before trusting scenario results.

| Layer | What it checks | Sections |
|-------|----------------|----------|
| **Structural** | 100% / 0% exit gates; triage rejection | 17–18 |
| **RNG control** | Fixed seed = identical; unseeded = variation | 19 |
| **Automated V&V** | Mass balance, drain, infinite capacity, demand stress | 20–21 |

**Manual stress tests** (edit Section 2): zero assessment capacity; fixed durations; warm-up 0/1/2/3 years.


## 17. Boundary test — 100% triage rejection

**Purpose:** Verify no downstream activity when every referral is rejected at triage.

**Expected:** zero queue events, zero diagnoses, all exits at triage.


In [61]:
TRACE = False
experiment = Experiment(auditor=Audit(), triage_rejected=1.0)
test_results = single_run(experiment, rep=0, run_length=365, warmup_days=366)

print("=== 100% Triage Rejection Test ===")
for key in [
    "ARRIVED_TOTAL",
    "ARRIVED_REFERRAL",
    "FLOW_REFERRAL_ACCEPTED",
    "EXIT_REFERRAL_REJECTED",
    "ARRIVED_SCREENING",
    "CLINICAL_COMPLETED_TOTAL",
]:
    print(f"{key:30s}: {test_results[key]}")

assert test_results["EXIT_REFERRAL_REJECTED"] == test_results["ARRIVED_REFERRAL"]
assert test_results["FLOW_REFERRAL_ACCEPTED"] == 0
assert test_results["ARRIVED_SCREENING"] == 0
assert test_results["CLINICAL_COMPLETED_TOTAL"] == 0
print("\nAll assertions passed.")


=== 100% Triage Rejection Test ===
ARRIVED_TOTAL                 : 1301
ARRIVED_REFERRAL              : 1301
FLOW_REFERRAL_ACCEPTED        : 0
EXIT_REFERRAL_REJECTED        : 1301
ARRIVED_SCREENING             : 0
CLINICAL_COMPLETED_TOTAL      : 0

All assertions passed.


## 18. Stage boundary tests (100% / 0% exits)

**Purpose:** Verify branching logic at each gate.

Sets each gate to 100% or 0% exit probability in turn:

- Triage, screening, pre-assessment
- Assessment, further assessment, diagnostic outcome


In [62]:
TRACE = False

stage_configs = {
    'triage': {
        'param': 'triage_rejected',
        'arrived_key': 'ARRIVED_REFERRAL',
        'exit_key': 'EXIT_REFERRAL_REJECTED',
        'pass_key': 'FLOW_REFERRAL_ACCEPTED',
        'downstream_entry_key': 'ARRIVED_SCREENING',
    },
    'screening': {
        'param': 'screening_discharge',
        'arrived_key': 'ARRIVED_SCREENING',
        'exit_key': 'EXIT_SCREENING_DISCHARGED',
        'pass_key': 'FLOW_SCREENING_PASSED',
        'downstream_entry_key': 'ARRIVED_PRE_ASSESS',
        'service_key': 'SERVICE_SCREENING_COMPLETED',
    },
    'pre_assessment': {
        'param': 'pre_assessment_rejection',
        'arrived_key': 'ARRIVED_PRE_ASSESS',
        'exit_key': 'EXIT_PRE_ASSESS_REJECTED',
        'pass_key': 'FLOW_PRE_ASSESS_PASSED',
        'downstream_entry_key': 'ARRIVED_ASSESSMENT',
        'service_key': 'SERVICE_PRE_ASSESS_COMPLETED',
    },
    'assessment': {
        'param': 'pct_non_diag_at_assessment',
        'arrived_key': 'ARRIVED_ASSESSMENT',
        'exit_key': 'EXIT_ASSESSMENT_NON_DIAGNOSIS',
        'pass_key': 'FLOW_ASSESSMENT_PASSED',
        'downstream_entry_key': 'ARRIVED_FURTHER_ASSESS',
        'service_key': 'SERVICE_ASSESSMENT_COMPLETED',
    },
    'further_assessment': {
        'param': 'pct_non_diag_at_further_assessment',
        'arrived_key': 'ARRIVED_FURTHER_ASSESS',
        'exit_key': 'EXIT_FURTHER_NON_DIAGNOSIS',
        'pass_key': 'FLOW_FURTHER_ASSESS_PASSED',
        'downstream_entry_key': 'FLOW_DIAGNOSIS_CONFIRMED',
        'service_key': 'SERVICE_FURTHER_ASSESS_COMPLETED',
    },
}


def run_stage_boundary_suite(stage_configs, boundary_name, boundary_prob):
    for stage, cfg in stage_configs.items():
        print(f'Testing stage ({boundary_name}): {stage}')
        experiment = Experiment(auditor=Audit(), **{cfg['param']: boundary_prob})
        test_results = single_run(experiment, rep=0, run_length=365, warmup_days=365)
        base_key = cfg.get('service_key', cfg['arrived_key'])
        base_count = test_results[base_key]
        if boundary_prob == 1.0:
            assert test_results[cfg['exit_key']] == base_count
            assert test_results[cfg['pass_key']] == 0
            if stage == 'triage':
                assert test_results[cfg['downstream_entry_key']] == 0
        elif boundary_prob == 0.0:
            assert test_results[cfg['exit_key']] == 0
            assert base_count > 0
            assert test_results[cfg['pass_key']] == base_count
        else:
            raise ValueError('Boundary probability must be 1.0 or 0.0')


run_stage_boundary_suite(stage_configs, '100% exits', 1.0)
run_stage_boundary_suite(stage_configs, '0% exits', 0.0)
print('\n' + '=' * 60)
print(' SUCCESS: All stage boundary suite verification checks passed!')
print('=' * 60)


Testing stage (100% exits): triage
Testing stage (100% exits): screening
Testing stage (100% exits): pre_assessment
Testing stage (100% exits): assessment
Testing stage (100% exits): further_assessment
Testing stage (0% exits): triage
Testing stage (0% exits): screening
Testing stage (0% exits): pre_assessment
Testing stage (0% exits): assessment
Testing stage (0% exits): further_assessment

 SUCCESS: All stage boundary suite verification checks passed!


## 19. Seed control verification

**Purpose:** Confirm reproducibility settings work as intended.

| Test | Expected |
|------|----------|
| Fixed seed, same rep | Identical results |
| Unseeded runs | Variation across runs |


In [63]:
def compare_core_outputs(df_a, df_b):
    cols = [
        "ARRIVED_TOTAL",
        "ACCESS_REFERRAL_TO_PRE_ASSESSMENT_RTT_DAYS",
        "ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS",
    ]
    return df_a[cols].equals(df_b[cols])

print("=" * 70)
print(" RANDOM NUMBER CONTROL VERIFICATION")
print("=" * 70)

short_horizon = 365 * 2
exp1 = Experiment(auditor=Audit())
df_seeded_run1 = multiple_runs(exp1, n_reps=6, run_length=short_horizon, use_fixed_seed=True)
exp2 = Experiment(auditor=Audit())
df_seeded_run2 = multiple_runs(exp2, n_reps=6, run_length=short_horizon, use_fixed_seed=True)

print("\n[TEST 1] Fixed-Seed Reproducibility")
print(" -> SUCCESS" if compare_core_outputs(df_seeded_run1, df_seeded_run2)
      else " -> FAILURE: Fixed-seed runs differed")

exp3 = Experiment(auditor=Audit())
df_unseeded_run1 = multiple_runs(exp3, n_reps=6, run_length=short_horizon, use_fixed_seed=False)
exp4 = Experiment(auditor=Audit())
df_unseeded_run2 = multiple_runs(exp4, n_reps=6, run_length=short_horizon, use_fixed_seed=False)

print("\n[TEST 2] Unseeded Variation")
print(" -> SUCCESS" if not compare_core_outputs(df_unseeded_run1, df_unseeded_run2)
      else " -> FAILURE: Unseeded runs were identical")
print("=" * 70)


 RANDOM NUMBER CONTROL VERIFICATION

[TEST 1] Fixed-Seed Reproducibility
 -> SUCCESS

[TEST 2] Unseeded Variation
 -> SUCCESS


## 20. V&V test suite definitions

**Purpose:** Define automated verification suites (run in Section 21).

| Suite | Checks |
|-------|--------|
| Mass conservation | Arrivals = exits + in system |
| Cohort drain | `IN_SYSTEM_END = 0`; valid RTT sample counts |
| Infinite capacity | RTT converges when queues removed |
| Demand stress | 5 → 10 → 20 rpd: queue, RTT, utilisation increase monotonically |


## 21. Run the V&V automation engine

**Purpose:** Execute all verification suites defined in **Section 20**.

> **Expected result:** `ALL SYSTEMS VERIFIED [100% PASS]`

Run the cell below after any model changes.


In [65]:
print("\n" + "#" * 65)
print("  SYSTEM VERIFICATION & VALIDATION (V&V) AUTOMATION ENGINE")
print("#" * 65)

try:
    run_seed_verification()
    run_flow_conservation_verification()
    run_rtt_cohort_verification()
    run_math_convergence_verification()
    run_demand_stress_verification()
    print("\n" + "=" * 65)
    print(" FINAL AUDIT RESULT: ALL SYSTEMS VERIFIED [100% PASS]")
    print("=" * 65 + "\n")
except AssertionError as error:
    print("\n" + "!" * 65)
    print(" AUDIT CRITICAL FAILURE:", error)
    print("!" * 65)



#################################################################
  SYSTEM VERIFICATION & VALIDATION (V&V) AUTOMATION ENGINE
#################################################################

 SUITE: 1. SEED CONTROL & REPRODUCIBILITY VERIFICATION
 -> SUCCESS: Fixed seeds are deterministic and reproducible.

 SUITE: 2. PATIENT FLOW MASS-CONSERVATION VERIFICATION
  Arrived: 6509 | Exited: 6509 | In system: 0
 -> SUCCESS: Mass balance verified (arrivals = exits + in system).

 SUITE: 3. RTT COHORT COMPLETENESS VERIFICATION
  Diagnosis RTT samples: 2961 | Diagnoses confirmed: 2961 | In system: 0
 -> SUCCESS: Cohort RTT computed after full drain (zero backlog).

 SUITE: 4. WORKFORCE-HOURS MATHEMATICAL CONVERGENCE
  Assessment RTT: 0.291 d | Diagnosis RTT: 0.371 d | Delta: 0.055
 -> SUCCESS: Infinite-capacity scheduler validation passed.

 SUITE: 5. DEMAND STRESS TEST
   5 rpd: peak queue=4 | diagnosis RTT=0.4 d | utilisation=20.3%
  10 rpd: peak queue=412 | diagnosis RTT=44.7 d | utilisati

---

## Part F — Next steps

## 22. Calibration and experiments

**Purpose:** Ideas for extending this model beyond the baseline notebook.

| Direction | Action |
|-----------|--------|
| **Calibrate durations** | Replace triangular placeholders with NHS/local data |
| **Calibrate branching** | Fit `PCT_*` from service audit or literature |
| **Capacity scenarios** | Vary `WORKFORCE_HOURS_*` and compare RTT distributions |
| **Demand scenarios** | Change `REFERRALS_PER_DAY`; use `multiple_runs()` for CIs |
| **Extract to package** | Move classes to `adhd_simpy/` for unit tests and CI |
| **Sensitivity analysis** | One-at-a-time or factorial parameter sweeps |

> After parameter changes, re-run **Section 12** (single run), **Section 15** (baseline), and **Section 21** (V&V).
